In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from util import project_points, scale_intrinsics, inv2x2, eigh_2x2, load_cameras, build_covariance

In [ ]:
def gaussian_rasterization(pos, color, opacity_raw, sigma, c2w, H, W, fx, fy,
                           cx, cy, near=2e-3, far=100, pix_guard=64, 
                           T=16, min_conis=1e-6, chi_square_clip=9.21,
                           alpha_max=0.99, alpha_cutoff=1/255.):
    # Shape notation used below:
    # N0 - number of input gaussians
    # Nf - number of gaussians after frustum filtering
    # Nk - number of gaussians after filtering NaN and infinity
    # Ns - number of on-screen gaussians
    # K  - total number of gaussian-tile intersections
    # U  - number of non-empty tiles
    # G  - number of gaussians in the current tile
    # P  - number of pixels in the current tile, P <= T * T
    #
    # pos:         [N0, 3]
    # color:       [N0, 3]
    # opacity_raw: [N0] or [N0, 1]
    # sigma:       [N0, 3, 3]
    # c2w:         [4, 4]
    uv, x, y, z = project_points(pos, c2w, H, W, fx, fy, cx, cy)
    # uv: [N0, 2]
    # x, y, z: [N0]
    
    device = pos.device
    dt = pos.dtype
    
    u, v = uv[:, 0], uv[:, 1] # each [N0]
    frustum = (
        (u > -pix_guard) 
        & (u < W + pix_guard) 
        & (v > -pix_guard) 
        & (v < H + pix_guard) 
        & (z > near) 
        & (z < far)
    ) # [N0]
    
    uv = uv[frustum] # [Nf, 2]
    pos = pos[frustum] # [Nf, 3]
    color = color[frustum] # [Nf, 3]
    opacity = torch.sigmoid(opacity_raw[frustum]).squeeze(-1).clamp(0, 0.999) # [Nf]
    x = x[frustum] # [Nf]
    y = y[frustum] # [Nf]
    z = z[frustum] # [Nf]
    sigma = sigma[frustum] # [Nf, 3, 3]
    
    ## Project the covariance
    Rcw = c2w[:3, :3] # [3, 3]
    Rwc = Rcw.T # [3, 3]
    
    # Eq. 5
    J = torch.zeros((pos.shape[0], 2, 3), device=device, dtype=dt) # [Nf, 2, 3]
    J[:, 0, 0] = fx / z
    J[:, 1, 1] = fy / z
    J[:, 0, 2] = -fx * x / (z * z)
    J[:, 1, 2] = -fy * y / (z * z)
    
    sigma_camera = Rwc.unsqueeze(0) @ sigma @ Rwc.T.unsqueeze(0) # [Nf, 3, 3]
    sigma_uv = J @ sigma_camera @ J.transpose(1, 2) # [Nf, 2, 2]
            
    # Enforce symmetry
    sigma_uv = 0.5 * (sigma_uv + sigma_uv.transpose(1, 2)) # [Nf, 2, 2]
        
    # Clamp projected gaussian ellipse size
    evals, evecs = eigh_2x2(sigma_uv)
    # evals: [Nf, 2]
    # evecs: [Nf, 2, 2]
    evals = torch.clamp(evals, min=1e-6, max=1e4) # [Nf, 2]
        
    sigma_uv = (
        evecs
        @ torch.diag_embed(evals)
        @ evecs.transpose(1, 2)
    ) # [Nf, 2, 2]
        
    # Filter NaN and infinity.
    # sigma_uv.reshape(...): [Nf, 4]
    keep = torch.isfinite(
        sigma_uv.reshape(sigma_uv.shape[0], -1)
    ).all(dim=-1) # [Nf]
    
    uv = uv[keep] # [Nk, 2]
    pos = pos[keep] # [Nk, 3]
    color = color[keep] # [Nk, 3]
    opacity = opacity[keep] # [Nk]
    z = z[keep] # [Nk]
    sigma_uv = sigma_uv[keep] # [Nk, 2, 2]
    evals = evals[keep] # [Nk, 2]
    
    z, order = torch.sort(z, descending=False)
    # z: [Nk]
    # order: [Nk]
    uv = uv[order] # [Nk, 2]
    color = color[order] # [Nk, 3]
    opacity = opacity[order] # [Nk]
    sigma_uv = sigma_uv[order] # [Nk, 2, 2]
    evals = evals[order] # [Nk, 2]
    
    u = uv[:, 0] # [Nk]
    v = uv[:, 1] # [Nk]
    
    # Tiling
    major_variance = evals[:, 1].clamp_min(1e-12).clamp_max(1e4) # [Nk]
    radius = torch.ceil(3.0 * torch.sqrt(major_variance)).to(torch.int64) # [Nk]
    
    umin = torch.floor(u - radius).to(torch.int64)
    umax = torch.floor(u + radius).to(torch.int64)
    vmin = torch.floor(v - radius).to(torch.int64)
    vmax = torch.floor(v + radius).to(torch.int64)
    
    on_screen = (
        (umax >= 0)
        & (umin < W)
        & (vmax >= 0)
        & (vmin < H)
    ) # [Nk]
    if not on_screen.any():
        raise Exception("there are no gaussians on screen")
    
    u, v = u[on_screen], v[on_screen] # each [Ns]
    color = color[on_screen] # [Ns, 3]
    opacity = opacity[on_screen] # [Ns]
    sigma_uv = sigma_uv[on_screen] # [Ns, 2, 2]
    umin, umax = umin[on_screen], umax[on_screen] # each [Ns]
    vmin, vmax = vmin[on_screen], vmax[on_screen] # each [Ns]
    
    umin = umin.clamp(0, W - 1) # [Ns]
    umax = umax.clamp(0, W - 1) # [Ns]
    vmin = vmin.clamp(0, H - 1) # [Ns]
    vmax = vmax.clamp(0, H - 1) # [Ns]
    
    # Tile index for each AABB.
    umin_tile = (umin // T).to(torch.int64) # [Ns]
    umax_tile = (umax // T).to(torch.int64) # [Ns]
    vmin_tile = (vmin // T).to(torch.int64) # [Ns]
    vmax_tile = (vmax // T).to(torch.int64) # [Ns]
    
    # Number of tiles each gaussian intersects along each axis.
    # Example:
    # n_u = [2, 1, 3]
    # n_v = [3, 2, 1]
    n_u = umax_tile - umin_tile + 1 # [Ns]
    n_v = vmax_tile - vmin_tile + 1 # [Ns]
    
    # Build only the actual gaussian-tile intersections. This keeps memory
    # proportional to K = sum(n_u * n_v), instead of Ns * max_u * max_v.
    #
    # Example:
    # n_u                       = [2, 1, 3]
    # n_v                       = [3, 2, 1]
    # num_tiles_per_gaussian    = [6, 2, 3]
    # num_tile_intersections K  = 6 + 2 + 3 = 11
    num_tiles_per_gaussian = n_u * n_v # [Ns]
    num_gaussians = umin_tile.shape[0] # Python scalar, equals Ns
    num_tile_intersections = int(num_tiles_per_gaussian.sum().item()) # Python scalar K

    # repeat_interleave repeats each gaussian id by the number of tiles
    # intersected by that gaussian.
    #
    # Small example:
    # torch.arange(3)           = [0, 1, 2]
    # repeats                   = [2, 1, 3]
    # repeat_interleave result  = [0, 0, 1, 2, 2, 2]
    gaussian_ids = torch.repeat_interleave(
        torch.arange(num_gaussians, device=device, dtype=torch.int64), # [Ns]
        num_tiles_per_gaussian, # [Ns]
        output_size=num_tile_intersections,
    ) # [K]

    # cumsum gives the exclusive end of each gaussian's segment.
    # Subtracting the segment length converts the ends into starts.
    #
    # Small example:
    # num_tiles_per_gaussian       = [2, 1, 3]
    # cumsum                       = [2, 3, 6]
    # cumsum - segment lengths     = [0, 2, 3]
    # Therefore gaussian 0 starts at 0, gaussian 1 at 2, gaussian 2 at 3.
    starts_per_gaussian = torch.cumsum(
        num_tiles_per_gaussian,
        dim=0,
    ) # [Ns]
    starts_per_gaussian = (
        starts_per_gaussian - num_tiles_per_gaussian
    ) # [Ns]

    # local_tile_ids restarts from zero for each gaussian.
    #
    # Continuing the example:
    # gaussian_ids                        = [0, 0, 1, 2, 2, 2]
    # starts_per_gaussian[gaussian_ids]   = [0, 0, 2, 3, 3, 3]
    # torch.arange(K)                     = [0, 1, 2, 3, 4, 5]
    # local_tile_ids                      = [0, 1, 0, 0, 1, 2]
    local_tile_ids = (
        torch.arange(num_tile_intersections, device=device, dtype=torch.int64) # [K]
        - starts_per_gaussian[gaussian_ids] # [K]
    ) # [K]

    # v changes fastest inside each gaussian's tile rectangle.
    # n_v[gaussian_ids]: [K]
    local_tile_u = local_tile_ids // n_v[gaussian_ids] # [K]
    local_tile_v = local_tile_ids % n_v[gaussian_ids] # [K]
    flat_tile_u = umin_tile[gaussian_ids] + local_tile_u # [K]
    flat_tile_v = vmin_tile[gaussian_ids] + local_tile_v # [K]

    num_tiles_u = (W + T - 1) // T # Python scalar
    flat_tile_id = flat_tile_v * num_tiles_u + flat_tile_u # [K]

    tile_ids_1d, perm = torch.sort(
        flat_tile_id,
        stable=True,
    )
    gaussian_ids = gaussian_ids[perm]
    
    unique_tile_ids, nb_gaussian_per_tile = torch.unique_consecutive(
        tile_ids_1d,
        return_counts=True,
    )
    # unique_tile_ids: [U]
    # nb_gaussian_per_tile: [U]
    start = torch.zeros_like(unique_tile_ids) # [U]
    start[1:] = torch.cumsum(nb_gaussian_per_tile[:-1], dim=0) # [U - 1]
    end = start + nb_gaussian_per_tile # [U]
    
    inverse_covariance = inv2x2(sigma_uv) # [Ns, 2, 2]
    inverse_covariance[:, 0, 0] = torch.clamp(
        inverse_covariance[:, 0, 0],
        min=min_conis,
    ) # selected diagonal: [Ns]
    inverse_covariance[:, 1, 1] = torch.clamp(
        inverse_covariance[:, 1, 1],
        min=min_conis,
    ) # selected diagonal: [Ns]
    
    final_image = torch.zeros(
        (H * W, 3),
        device=device,
        dtype=dt,
    ) # [H * W, 3]
    
    # Iterate over non-empty tiles.
    # unique_tile_ids.tolist(), start.tolist(), end.tolist(): each Python list [U]
    for tile_id, s0, s1 in zip(
        unique_tile_ids.tolist(),
        start.tolist(),
        end.tolist(),
    ):
        txi = tile_id % num_tiles_u # Python scalar
        tyi = tile_id // num_tiles_u # Python scalar
        
        tile_gaussian_ids = gaussian_ids[s0:s1] # [G]
        
        x0, y0 = txi * T, tyi * T # Python scalars
        x1 = min((txi + 1) * T, W) # Python scalar
        y1 = min((tyi + 1) * T, H) # Python scalar
        if x0 >= x1 or y0 >= y1:
            continue
        
        xs = torch.arange(x0, x1, device=device, dtype=dt) # [tile_width]
        ys = torch.arange(y0, y1, device=device, dtype=dt) # [tile_height]
        pu, pv = torch.meshgrid(xs, ys, indexing='xy')
        # pu, pv: [tile_height, tile_width]
        px_u = pu.reshape(-1) # [P]
        px_v = pv.reshape(-1) # [P]
        
        pixel_idx_1d = (px_v * W + px_u).to(torch.int64) # [P]
        
        gaussian_i_u = u[tile_gaussian_ids] # [G]
        gaussian_i_v = v[tile_gaussian_ids] # [G]
        gaussian_i_color = color[tile_gaussian_ids] # [G, 3]
        gaussian_i_opacity = opacity[tile_gaussian_ids] # [G]
        gaussian_i_inverse_covariance = inverse_covariance[tile_gaussian_ids] # [G, 2, 2]
        
        du = px_u.unsqueeze(0) - gaussian_i_u.unsqueeze(-1) # [G, P]
        dv = px_v.unsqueeze(0) - gaussian_i_v.unsqueeze(-1) # [G, P]
        
        A11 = gaussian_i_inverse_covariance[:, 0, 0].unsqueeze(-1) # [G, 1]
        A12 = gaussian_i_inverse_covariance[:, 0, 1].unsqueeze(-1) # [G, 1]
        A22 = gaussian_i_inverse_covariance[:, 1, 1].unsqueeze(-1) # [G, 1]
        q = A11 * du * du + 2 * A12 * du * dv + A22 * dv * dv # [G, P]
        
        inside = q <= chi_square_clip # [G, P]
        g = torch.exp(-0.5 * torch.clamp(q, max=chi_square_clip)) # [G, P]
        g = torch.where(inside, g, torch.zeros_like(g)) # [G, P]
                
        alpha_i = (gaussian_i_opacity.unsqueeze(-1) * g).clamp_max(alpha_max) # [G, P]
        alpha_i = torch.where(alpha_i >= alpha_cutoff, alpha_i, torch.zeros_like(alpha_i),) # [G, P]
        one_minus_alpha_i = 1 - alpha_i # [G, P]
        T_i = torch.cumprod(one_minus_alpha_i, dim=0) # [G, P]
        T_i = torch.concatenate([
            torch.ones((1, alpha_i.shape[-1]), device=device, dtype=dt), # [1, P]
            T_i[:-1], # [G - 1, P]
        ], dim=0) # [G, P]
                
        w = alpha_i * T_i # [G, P]
        tile_color = (
            w.unsqueeze(-1) # [G, P, 1]
            * gaussian_i_color.unsqueeze(1) # [G, 1, 3]
        ).sum(dim=0) # [P, 3]
        
        final_image[pixel_idx_1d] = tile_color
        # final_image[pixel_idx_1d]: [P, 3]
        
    return final_image.reshape((H, W, 3)).clamp(0, 1) # [H, W, 3]


## Handwritten Metal renderer

Projection, tile assignment and depth sorting remain in PyTorch. Only the per-tile raster loop is a handwritten Metal compute shader compiled at runtime with `torch.mps.compile_shader`, so no C++ extension or Xcode project is required.


In [ ]:
import time
from functools import lru_cache


_last_metal_counts_per_tile = None
_last_metal_kernel_seconds = None
_last_metal_source = None


@lru_cache(maxsize=None)
def build_metal_tile_rasterizer(
    height,
    width,
    tile_size,
    chi_square_clip,
    alpha_max,
    alpha_cutoff,
):
    """
    Compile a handwritten Metal kernel with one threadgroup per image tile and
    one thread per pixel. PyTorch owns all buffers and dispatches the shader.
    """
    if not torch.backends.mps.is_available():
        raise RuntimeError("The Metal renderer requires an available MPS device")

    threads_per_tile = tile_size * tile_size
    if threads_per_tile > 256:
        raise ValueError("This first Metal version supports at most 256 pixels per tile")

    num_tiles_u = (width + tile_size - 1) // tile_size
    gaussian_components = 9

    source = r"""
#include <metal_stdlib>
using namespace metal;

#define IMAGE_WIDTH __IMAGE_WIDTH__
#define IMAGE_HEIGHT __IMAGE_HEIGHT__
#define TILE_SIZE __TILE_SIZE__
#define NUM_TILES_U __NUM_TILES_U__
#define THREADS_PER_TILE __THREADS_PER_TILE__
#define GAUSSIAN_COMPONENTS __GAUSSIAN_COMPONENTS__
#define CHI_SQUARE_CLIP __CHI_SQUARE_CLIP__f
#define ALPHA_MAX __ALPHA_MAX__f
#define ALPHA_CUTOFF __ALPHA_CUTOFF__f

kernel void tile_rasterizer_kernel(
    const device float* gaussians [[buffer(0)]],
    const device int* gaussian_ids [[buffer(1)]],
    const device int* tile_offsets [[buffer(2)]],
    device float* final_image [[buffer(3)]],
    uint tile_id [[threadgroup_position_in_grid]],
    uint pixel_in_tile [[thread_index_in_threadgroup]]
) {
    threadgroup float gaussians_shared[
        THREADS_PER_TILE * GAUSSIAN_COMPONENTS
    ];

    const uint tile_x = tile_id % NUM_TILES_U;
    const uint tile_y = tile_id / NUM_TILES_U;
    const uint pixel_x = tile_x * TILE_SIZE + pixel_in_tile % TILE_SIZE;
    const uint pixel_y = tile_y * TILE_SIZE + pixel_in_tile / TILE_SIZE;
    const bool valid_pixel = pixel_x < IMAGE_WIDTH && pixel_y < IMAGE_HEIGHT;

    const int tile_start = tile_offsets[tile_id];
    const int tile_end = tile_offsets[tile_id + 1];

    float transmittance = 1.0f;
    float3 accumulated_color = float3(0.0f);

    for (
        int batch_start = tile_start;
        batch_start < tile_end;
        batch_start += THREADS_PER_TILE
    ) {
        const int batch_count = min(
            THREADS_PER_TILE,
            tile_end - batch_start
        );

        if (int(pixel_in_tile) < batch_count) {
            const int gaussian_id =
                gaussian_ids[batch_start + int(pixel_in_tile)];
            const int source_base = gaussian_id * GAUSSIAN_COMPONENTS;
            const int shared_base =
                int(pixel_in_tile) * GAUSSIAN_COMPONENTS;

            #pragma clang loop unroll(full)
            for (int component = 0; component < GAUSSIAN_COMPONENTS; ++component) {
                gaussians_shared[shared_base + component] =
                    gaussians[source_base + component];
            }
        }

        threadgroup_barrier(mem_flags::mem_threadgroup);

        if (valid_pixel) {
            for (
                int local_gaussian_id = 0;
                local_gaussian_id < batch_count;
                ++local_gaussian_id
            ) {
                const int gaussian_base =
                    local_gaussian_id * GAUSSIAN_COMPONENTS;
                const float du =
                    float(pixel_x) - gaussians_shared[gaussian_base];
                const float dv =
                    float(pixel_y) - gaussians_shared[gaussian_base + 1];

                const float q =
                    gaussians_shared[gaussian_base + 6] * du * du
                    + 2.0f * gaussians_shared[gaussian_base + 7] * du * dv
                    + gaussians_shared[gaussian_base + 8] * dv * dv;

                if (q <= CHI_SQUARE_CLIP) {
                    const float alpha = min(
                        gaussians_shared[gaussian_base + 5]
                            * exp(-0.5f * q),
                        ALPHA_MAX
                    );

                    if (alpha >= ALPHA_CUTOFF) {
                        const float weight = transmittance * alpha;
                        accumulated_color += weight * float3(
                            gaussians_shared[gaussian_base + 2],
                            gaussians_shared[gaussian_base + 3],
                            gaussians_shared[gaussian_base + 4]
                        );
                        transmittance *= 1.0f - alpha;
                    }
                }
            }
        }

        threadgroup_barrier(mem_flags::mem_threadgroup);
    }

    if (valid_pixel) {
        const uint output_base =
            (pixel_y * IMAGE_WIDTH + pixel_x) * 3;
        final_image[output_base] = accumulated_color.x;
        final_image[output_base + 1] = accumulated_color.y;
        final_image[output_base + 2] = accumulated_color.z;
    }
}
"""

    replacements = {
        "__IMAGE_WIDTH__": str(width),
        "__IMAGE_HEIGHT__": str(height),
        "__TILE_SIZE__": str(tile_size),
        "__NUM_TILES_U__": str(num_tiles_u),
        "__THREADS_PER_TILE__": str(threads_per_tile),
        "__GAUSSIAN_COMPONENTS__": str(gaussian_components),
        "__CHI_SQUARE_CLIP__": repr(float(chi_square_clip)),
        "__ALPHA_MAX__": repr(float(alpha_max)),
        "__ALPHA_CUTOFF__": repr(float(alpha_cutoff)),
    }
    for placeholder, value in replacements.items():
        source = source.replace(placeholder, value)

    library = torch.mps.compile_shader(source)
    return library.tile_rasterizer_kernel, source


def gaussian_rasterization_metal(pos, color, opacity_raw, sigma, c2w, H, W, fx, fy,
                           cx, cy, near=2e-3, far=100, pix_guard=64, 
                           T=16, min_conis=1e-6, chi_square_clip=9.21,
                           alpha_max=0.99, alpha_cutoff=1/255.):
    # Shape notation used below:
    # N0 - number of input gaussians
    # Nf - number of gaussians after frustum filtering
    # Nk - number of gaussians after filtering NaN and infinity
    # Ns - number of on-screen gaussians
    # K  - total number of gaussian-tile intersections
    # U  - number of non-empty tiles
    # G  - number of gaussians in the current tile
    # P  - number of pixels in the current tile, P <= T * T
    #
    # pos:         [N0, 3]
    # color:       [N0, 3]
    # opacity_raw: [N0] or [N0, 1]
    # sigma:       [N0, 3, 3]
    # c2w:         [4, 4]
    uv, x, y, z = project_points(pos, c2w, H, W, fx, fy, cx, cy)
    # uv: [N0, 2]
    # x, y, z: [N0]
    
    device = pos.device
    dt = pos.dtype
    
    u, v = uv[:, 0], uv[:, 1] # each [N0]
    frustum = (
        (u > -pix_guard) 
        & (u < W + pix_guard) 
        & (v > -pix_guard) 
        & (v < H + pix_guard) 
        & (z > near) 
        & (z < far)
    ) # [N0]
    
    uv = uv[frustum] # [Nf, 2]
    pos = pos[frustum] # [Nf, 3]
    color = color[frustum] # [Nf, 3]
    opacity = torch.sigmoid(opacity_raw[frustum]).squeeze(-1).clamp(0, 0.999) # [Nf]
    x = x[frustum] # [Nf]
    y = y[frustum] # [Nf]
    z = z[frustum] # [Nf]
    sigma = sigma[frustum] # [Nf, 3, 3]
    
    ## Project the covariance
    Rcw = c2w[:3, :3] # [3, 3]
    Rwc = Rcw.T # [3, 3]
    
    # Eq. 5
    J = torch.zeros((pos.shape[0], 2, 3), device=device, dtype=dt) # [Nf, 2, 3]
    J[:, 0, 0] = fx / z
    J[:, 1, 1] = fy / z
    J[:, 0, 2] = -fx * x / (z * z)
    J[:, 1, 2] = -fy * y / (z * z)
    
    sigma_camera = Rwc.unsqueeze(0) @ sigma @ Rwc.T.unsqueeze(0) # [Nf, 3, 3]
    sigma_uv = J @ sigma_camera @ J.transpose(1, 2) # [Nf, 2, 2]
            
    # Enforce symmetry
    sigma_uv = 0.5 * (sigma_uv + sigma_uv.transpose(1, 2)) # [Nf, 2, 2]
        
    # Clamp projected gaussian ellipse size
    evals, evecs = eigh_2x2(sigma_uv)
    # evals: [Nf, 2]
    # evecs: [Nf, 2, 2]
    evals = torch.clamp(evals, min=1e-6, max=1e4) # [Nf, 2]
        
    sigma_uv = (
        evecs
        @ torch.diag_embed(evals)
        @ evecs.transpose(1, 2)
    ) # [Nf, 2, 2]
        
    # Filter NaN and infinity.
    # sigma_uv.reshape(...): [Nf, 4]
    keep = torch.isfinite(
        sigma_uv.reshape(sigma_uv.shape[0], -1)
    ).all(dim=-1) # [Nf]
    
    uv = uv[keep] # [Nk, 2]
    pos = pos[keep] # [Nk, 3]
    color = color[keep] # [Nk, 3]
    opacity = opacity[keep] # [Nk]
    z = z[keep] # [Nk]
    sigma_uv = sigma_uv[keep] # [Nk, 2, 2]
    evals = evals[keep] # [Nk, 2]
    
    z, order = torch.sort(z, descending=False)
    # z: [Nk]
    # order: [Nk]
    uv = uv[order] # [Nk, 2]
    color = color[order] # [Nk, 3]
    opacity = opacity[order] # [Nk]
    sigma_uv = sigma_uv[order] # [Nk, 2, 2]
    evals = evals[order] # [Nk, 2]
    
    u = uv[:, 0] # [Nk]
    v = uv[:, 1] # [Nk]
    
    # Tiling
    major_variance = evals[:, 1].clamp_min(1e-12).clamp_max(1e4) # [Nk]
    radius = torch.ceil(3.0 * torch.sqrt(major_variance)).to(torch.int64) # [Nk]
    
    umin = torch.floor(u - radius).to(torch.int64)
    umax = torch.floor(u + radius).to(torch.int64)
    vmin = torch.floor(v - radius).to(torch.int64)
    vmax = torch.floor(v + radius).to(torch.int64)
    
    on_screen = (
        (umax >= 0)
        & (umin < W)
        & (vmax >= 0)
        & (vmin < H)
    ) # [Nk]
    if not on_screen.any():
        raise Exception("there are no gaussians on screen")
    
    u, v = u[on_screen], v[on_screen] # each [Ns]
    color = color[on_screen] # [Ns, 3]
    opacity = opacity[on_screen] # [Ns]
    sigma_uv = sigma_uv[on_screen] # [Ns, 2, 2]
    umin, umax = umin[on_screen], umax[on_screen] # each [Ns]
    vmin, vmax = vmin[on_screen], vmax[on_screen] # each [Ns]
    
    umin = umin.clamp(0, W - 1) # [Ns]
    umax = umax.clamp(0, W - 1) # [Ns]
    vmin = vmin.clamp(0, H - 1) # [Ns]
    vmax = vmax.clamp(0, H - 1) # [Ns]
    
    # Tile index for each AABB.
    umin_tile = (umin // T).to(torch.int64) # [Ns]
    umax_tile = (umax // T).to(torch.int64) # [Ns]
    vmin_tile = (vmin // T).to(torch.int64) # [Ns]
    vmax_tile = (vmax // T).to(torch.int64) # [Ns]
    
    # Number of tiles each gaussian intersects along each axis.
    # Example:
    # n_u = [2, 1, 3]
    # n_v = [3, 2, 1]
    n_u = umax_tile - umin_tile + 1 # [Ns]
    n_v = vmax_tile - vmin_tile + 1 # [Ns]
    
    # Build only the actual gaussian-tile intersections. This keeps memory
    # proportional to K = sum(n_u * n_v), instead of Ns * max_u * max_v.
    #
    # Example:
    # n_u                       = [2, 1, 3]
    # n_v                       = [3, 2, 1]
    # num_tiles_per_gaussian    = [6, 2, 3]
    # num_tile_intersections K  = 6 + 2 + 3 = 11
    num_tiles_per_gaussian = n_u * n_v # [Ns]
    num_gaussians = umin_tile.shape[0] # Python scalar, equals Ns
    num_tile_intersections = int(num_tiles_per_gaussian.sum().item()) # Python scalar K

    # repeat_interleave repeats each gaussian id by the number of tiles
    # intersected by that gaussian.
    #
    # Small example:
    # torch.arange(3)           = [0, 1, 2]
    # repeats                   = [2, 1, 3]
    # repeat_interleave result  = [0, 0, 1, 2, 2, 2]
    gaussian_ids = torch.repeat_interleave(
        torch.arange(num_gaussians, device=device, dtype=torch.int64), # [Ns]
        num_tiles_per_gaussian, # [Ns]
        output_size=num_tile_intersections,
    ) # [K]

    # cumsum gives the exclusive end of each gaussian's segment.
    # Subtracting the segment length converts the ends into starts.
    #
    # Small example:
    # num_tiles_per_gaussian       = [2, 1, 3]
    # cumsum                       = [2, 3, 6]
    # cumsum - segment lengths     = [0, 2, 3]
    # Therefore gaussian 0 starts at 0, gaussian 1 at 2, gaussian 2 at 3.
    starts_per_gaussian = torch.cumsum(
        num_tiles_per_gaussian,
        dim=0,
    ) # [Ns]
    starts_per_gaussian = (
        starts_per_gaussian - num_tiles_per_gaussian
    ) # [Ns]

    # local_tile_ids restarts from zero for each gaussian.
    #
    # Continuing the example:
    # gaussian_ids                        = [0, 0, 1, 2, 2, 2]
    # starts_per_gaussian[gaussian_ids]   = [0, 0, 2, 3, 3, 3]
    # torch.arange(K)                     = [0, 1, 2, 3, 4, 5]
    # local_tile_ids                      = [0, 1, 0, 0, 1, 2]
    local_tile_ids = (
        torch.arange(num_tile_intersections, device=device, dtype=torch.int64) # [K]
        - starts_per_gaussian[gaussian_ids] # [K]
    ) # [K]

    # v changes fastest inside each gaussian's tile rectangle.
    # n_v[gaussian_ids]: [K]
    local_tile_u = local_tile_ids // n_v[gaussian_ids] # [K]
    local_tile_v = local_tile_ids % n_v[gaussian_ids] # [K]
    flat_tile_u = umin_tile[gaussian_ids] + local_tile_u # [K]
    flat_tile_v = vmin_tile[gaussian_ids] + local_tile_v # [K]

    num_tiles_u = (W + T - 1) // T # Python scalar
    flat_tile_id = flat_tile_v * num_tiles_u + flat_tile_u # [K]

    tile_ids_1d, perm = torch.sort(
        flat_tile_id,
        stable=True,
    )
    gaussian_ids = gaussian_ids[perm]
    
    unique_tile_ids, nb_gaussian_per_tile = torch.unique_consecutive(
        tile_ids_1d,
        return_counts=True,
    )
    # unique_tile_ids: [U]
    # nb_gaussian_per_tile: [U]
    start = torch.zeros_like(unique_tile_ids) # [U]
    start[1:] = torch.cumsum(nb_gaussian_per_tile[:-1], dim=0) # [U - 1]
    end = start + nb_gaussian_per_tile # [U]
    
    inverse_covariance = inv2x2(sigma_uv) # [Ns, 2, 2]
    inverse_covariance[:, 0, 0] = torch.clamp(
        inverse_covariance[:, 0, 0],
        min=min_conis,
    ) # selected diagonal: [Ns]
    inverse_covariance[:, 1, 1] = torch.clamp(
        inverse_covariance[:, 1, 1],
        min=min_conis,
    ) # selected diagonal: [Ns]
    
    if dt != torch.float32:
        raise TypeError("The first Metal version expects float32 Gaussian data")

    num_tiles_v = (H + T - 1) // T
    num_tiles = num_tiles_u * num_tiles_v

    # Dense offsets keep the Metal dispatch grid independent of the number
    # of non-empty tiles. Empty tiles simply have equal start/end offsets.
    counts_per_tile = torch.zeros(
        num_tiles,
        device=device,
        dtype=torch.int32,
    )
    counts_per_tile[unique_tile_ids] = nb_gaussian_per_tile.to(torch.int32)

    tile_offsets = torch.zeros(
        num_tiles + 1,
        device=device,
        dtype=torch.int32,
    )
    tile_offsets[1:] = torch.cumsum(counts_per_tile, dim=0)

    centers = torch.stack((u, v), dim=-1)
    conics = torch.stack(
        (
            inverse_covariance[:, 0, 0],
            inverse_covariance[:, 0, 1],
            inverse_covariance[:, 1, 1],
        ),
        dim=-1,
    )
    # [u, v, r, g, b, opacity, A11, A12, A22]
    gaussians = torch.cat(
        (
            centers,
            color,
            opacity.unsqueeze(-1),
            conics,
        ),
        dim=-1,
    ).to(torch.float32).contiguous()
    gaussian_ids = gaussian_ids.to(torch.int32).contiguous()

    final_image = torch.empty(
        (H, W, 3),
        device=device,
        dtype=torch.float32,
    )

    tile_rasterizer, metal_source = build_metal_tile_rasterizer(
        H,
        W,
        T,
        float(chi_square_clip),
        float(alpha_max),
        float(alpha_cutoff),
    )
    global _last_metal_counts_per_tile
    global _last_metal_kernel_seconds
    global _last_metal_source

    _last_metal_counts_per_tile = counts_per_tile.detach()
    _last_metal_source = metal_source

    if device.type == "mps":
        torch.mps.synchronize()
    kernel_started_at = time.perf_counter()

    threads_per_tile = T * T
    tile_rasterizer(
        gaussians,
        gaussian_ids,
        tile_offsets,
        final_image,
        threads=[num_tiles * threads_per_tile, 1, 1],
        group_size=[threads_per_tile, 1, 1],
    )

    if device.type == "mps":
        torch.mps.synchronize()
    _last_metal_kernel_seconds = time.perf_counter() - kernel_started_at

    return final_image.clamp(0, 1) # [H, W, 3]



In [ ]:
scene = "bonsai"

device = torch.device("mps")

pos = torch.from_numpy(torch.load('out_bonsai/pos_param.pt', weights_only=False)).to(device)
opacity_raw = torch.from_numpy(torch.load('out_bonsai/alpha_raw_param.pt', weights_only=False)).to(device)
color = torch.sigmoid(0.282 * torch.from_numpy(torch.load('out_bonsai/f_dc.pt', weights_only=False))).to(device)
scale_raw = torch.from_numpy(torch.load('out_bonsai/scale_raw.pt', weights_only=False)).to(device)
rot_raw = torch.from_numpy(torch.load('out_bonsai/rot_raw.pt', weights_only=False)).to(device)

sigma = build_covariance(scale_raw, rot_raw)

cam_parameters = np.load(f'out_colmap/{scene}/cam_meta.npy', allow_pickle=True).item()

H = cam_parameters['height']
W = cam_parameters['width']
fx, fy = cam_parameters['fx'], cam_parameters['fy']
cx, cy = W / 2, H / 2

H_scaled = H // 2
W_scaled = W // 2

fx, fy, cx, cy = scale_intrinsics(H_scaled, W_scaled, H, W, fx, fy, cx, cy)

H = H_scaled
W = W_scaled

c2ws, images_paths = load_cameras(f'out_colmap/{scene}/cameras.npy', f'image_data/{scene}/images_2')

CAM_ID = 10

c2w = c2ws[CAM_ID].to(device)
image_path = images_paths[CAM_ID]

img = gaussian_rasterization(pos, color, opacity_raw, sigma, c2w, H, W, fx, fy, cx, cy)

In [ ]:
Image.fromarray((img.cpu().detach().numpy() * 255).astype(np.uint8))

## PyTorch vs handwritten Metal

The Metal shader is warmed up before timing. Both renderers receive the same inputs, display their images, and are compared value by value.


In [ ]:
renderer_args = (
    pos,
    color,
    opacity_raw,
    sigma,
    c2w,
    H,
    W,
    fx,
    fy,
    cx,
    cy,
)


def synchronize_device():
    if device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "cuda":
        torch.cuda.synchronize(device)


def measure_renderer(renderer, *args):
    synchronize_device()
    started_at = time.perf_counter()
    image = renderer(*args)
    synchronize_device()
    elapsed_seconds = time.perf_counter() - started_at
    return image, elapsed_seconds


# Compile and warm up the Metal kernel before comparing execution time.
_ = gaussian_rasterization_metal(*renderer_args)
synchronize_device()

img_pytorch, pytorch_seconds = measure_renderer(
    gaussian_rasterization,
    *renderer_args,
)
img_metal, metal_seconds = measure_renderer(
    gaussian_rasterization_metal,
    *renderer_args,
)

print(f"PyTorch renderer: {pytorch_seconds * 1_000:.2f} ms")
print(f"Metal renderer: {metal_seconds * 1_000:.2f} ms")


In [ ]:
counts_cpu = _last_metal_counts_per_tile.cpu()

print("Metal diagnostic")
print(f"  kernel only: {_last_metal_kernel_seconds * 1_000:.2f} ms")
print(f"  compiled kernels in lru_cache: {build_metal_tile_rasterizer.cache_info().currsize}")
print(f"  image tiles: {counts_cpu.numel()}")
print(f"  non-empty tiles: {(counts_cpu > 0).sum().item()}")
print(f"  Gaussian-tile intersections: {counts_cpu.sum().item()}")
print(f"  Gaussians per tile, mean: {counts_cpu.float().mean().item():.2f}")
print(f"  Gaussians per tile, median: {counts_cpu.float().median().item():.0f}")
print(f"  Gaussians per tile, p95: {torch.quantile(counts_cpu.float(), 0.95).item():.0f}")
print(f"  Gaussians per tile, max: {counts_cpu.max().item()}")
print(f"  handwritten Metal source: {len(_last_metal_source.splitlines())} lines")

# Uncomment when the complete handwritten MSL is needed:
# print(_last_metal_source)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

axes[0].imshow(img_pytorch.detach().cpu().numpy())
axes[0].set_title("PyTorch tile loop")
axes[0].axis("off")

axes[1].imshow(img_metal.detach().cpu().numpy())
axes[1].set_title("Handwritten Metal tile loop")
axes[1].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
absolute_difference = (img_pytorch - img_metal).abs()

strict_atol = 5e-5
metal_atol = 1e-3
maximum_strict_mismatch_fraction = 1e-5

max_absolute_difference = absolute_difference.max().item()
mean_absolute_difference = absolute_difference.mean().item()
strict_mismatches = (absolute_difference > strict_atol).sum().item()
strict_mismatch_fraction = strict_mismatches / absolute_difference.numel()

print(f"Maximum absolute difference: {max_absolute_difference:.8f}")
print(f"Mean absolute difference: {mean_absolute_difference:.8f}")
print(f"Values with absolute difference > {strict_atol}: {strict_mismatches}")
print(f"Strict mismatch fraction: {strict_mismatch_fraction:.10%}")

# A value close to q == chi_square_clip or alpha == alpha_cutoff can land on
# opposite sides of the threshold because PyTorch MPS and the handwritten
# Metal shader evaluate float32 expressions with slightly different rounding.
# Keep the original 5e-5 threshold as a diagnostic and reject systematic
# disagreement, while allowing isolated boundary values up to 1e-3.
assert strict_mismatch_fraction <= maximum_strict_mismatch_fraction, (
    "Too many pixels differ at the strict 5e-5 threshold"
)

torch.testing.assert_close(
    img_metal,
    img_pytorch,
    rtol=1e-4,
    atol=metal_atol,
)
print("Pixel-wise comparison passed within float32 Metal tolerance.")

plt.figure(figsize=(8, 6))
plt.imshow(absolute_difference.max(dim=-1).values.detach().cpu().numpy())
plt.title("Maximum absolute difference per pixel")
plt.colorbar()
plt.axis("off")
plt.show()
